# Confetti Tube Production — Discrete-Event Simulation Analysis
**Course:** Simulation Modeling and Analysis (ESMA341) — Khalifa University, Fall 2025  
**Tool:** Simio (model files: `Current_State_Final_Model.spfx`, `Future_State_Final_Model.spfx`)  

## Problem Statement
A confetti tube production facility manufactures three product types (Small, Large, XL) across a 9-stage flow line. Management suspects the system wastes time due to congestion and blocking, fails to meet daily demand, and generates excessive work-in-process (WIP) inventory.

**Objective:** Build a discrete-event simulation model to (1) quantify current performance and identify bottlenecks, and (2) evaluate improvement scenarios via a future-state model.

## Production Flow
```
Rolling → Trimming → ChemicalPressing → ChemicalMixing → BodyPreparation
       → CNC → GlueAndDrying → FinalAssembly → LabelingAndPackaging → FinishedGoods
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

## 1. Model Parameters

In [ ]:
# Simulation configuration
shift_minutes  = 480   # 8 hours/day (7-12, 1-4)
run_days       = 60    # simulation horizon
replications   = 15    # for statistical reliability
confidence     = 0.95

# Seasonal demand scenarios
demand = {
    'Fall':   200,  # tubes/day
    'Spring': 300,
    'Winter':  40
}

# Raw material restocking
materials = {
    'Paper Tubes':        {'interval': '12 weeks (quarterly)', 'lot_size': 5000},
    'Chemicals':          {'interval': '1 week',               'lot_size': 500},
    'Hardware Units':     {'interval': '1 week',               'lot_size': 300},
    'PVC Boards':         {'interval': '4 weeks (monthly)',    'lot_size': 400},
    'Assembly Fixtures':  {'interval': '8 weeks (bi-monthly)', 'lot_size': 1000},
}

print("=== Simulation Configuration ===")
print(f"Shift length:    {shift_minutes} min/day ({shift_minutes/60:.0f} hours)")
print(f"Run length:      {run_days} days")
print(f"Replications:    {replications}")
print(f"\nSeasonal demand scenarios:")
for season, d in demand.items():
    print(f"  {season:<8}: {d} tubes/day")

print(f"\nRaw material restocking:")
for mat, info in materials.items():
    print(f"  {mat:<22}: {info['lot_size']} units every {info['interval']}")

## 2. Service Times (Triangular Distributions)
Estimated from GEMBA walk videos. Parameters: Triangular(min, mode, max) in minutes.

In [ ]:
# Service time parameters (min, mode, max) in minutes by station and tube type
# Estimated from instructional videos as per project methodology
service_times = {
    'Rolling':             {'Small': (1.5, 2.0, 4.0), 'Large': (2.5, 3.5, 5.0), 'XL': (3.0, 4.5, 6.0)},
    'Trimming':            {'Small': (1.0, 1.5, 3.0), 'Large': (2.0, 3.0, 4.5), 'XL': (2.5, 3.5, 5.0)},
    'ChemicalPressing':    {'Small': (2.0, 3.0, 5.0), 'Large': (3.0, 4.0, 6.0), 'XL': (3.5, 5.0, 7.0)},
    'ChemicalMixing':      {'Small': (3.0, 4.5, 7.0), 'Large': (4.0, 6.0, 8.0), 'XL': (5.0, 7.0, 9.0)},
    'BodyPreparation':     {'Small': (2.0, 3.0, 5.0), 'Large': (3.0, 4.0, 6.0), 'XL': (3.5, 5.0, 7.0)},
    'CNC':                 {'Small': (2.5, 4.0, 6.0), 'Large': (3.5, 5.0, 7.0), 'XL': (4.0, 6.0, 8.0)},
    'GlueAndDrying':       {'Small': (3.0, 5.0, 8.0), 'Large': (4.0, 6.0, 9.0), 'XL': (5.0, 7.0,10.0)},
    'FinalAssembly':       {'Small': (2.5, 4.0, 6.0), 'Large': (3.5, 5.0, 7.0), 'XL': (4.0, 6.0, 8.0)},
    'LabelingAndPackaging':{'Small': (2.0, 3.5, 5.5), 'Large': (3.0, 4.5, 6.5), 'XL': (3.5, 5.0, 7.0)},
}

# Compute triangular mean = (a + b + c) / 3
rows = []
for station, types in service_times.items():
    row = {'Station': station}
    for tube_type, (a, b, c) in types.items():
        row[f'{tube_type} mean (min)'] = round((a + b + c) / 3, 2)
    rows.append(row)

df_times = pd.DataFrame(rows)
print("Mean service times by station and tube type (minutes):")
print(df_times.to_string(index=False))

## 3. Current State Results — Throughput vs Demand

In [ ]:
# Results from 15-replication Simio experiment (from report)
current_throughput = 20  # tubes/day — consistent across all seasons

results_throughput = pd.DataFrame({
    'Season': ['Fall', 'Spring', 'Winter'],
    'Demand (tubes/day)': [200, 300, 40],
    'Throughput (tubes/day)': [current_throughput] * 3,
    'Demand Met (%)': [round(current_throughput/d*100, 1) for d in [200, 300, 40]]
})

print("Current State — Throughput vs Demand:")
print(results_throughput.to_string(index=False))
print(f"\nKey finding: throughput is ~{current_throughput} tubes/day REGARDLESS of demand season.")
print("This confirms a hard capacity bottleneck, not a demand-side issue.")

## 4. Resource Utilization — Bottleneck Identification

In [ ]:
# Scheduled utilization from Simio experiment results (from report)
utilization_data = {
    'Station': ['Rolling', 'Trimming', 'ChemicalPressing', 'ChemicalMixing',
                'BodyPreparation', 'CNC', 'GlueAndDrying', 'FinalAssembly', 'LabelingAndPackaging'],
    'Fall (%)':   [82, 80, 85, 100, 83, 96.1, 99.1, 91, 95],
    'Spring (%)': [84, 82, 87, 100, 85, 95.7, 99.1, 92, 96],
    'Winter (%)': [75, 73, 78, 100, 76, 94.1, 96.3, 87, 90]
}

df_util = pd.DataFrame(utilization_data)
print("Scheduled Utilization by Station and Season (%)")
print(df_util.to_string(index=False))
print("\nPrimary bottleneck:   ChemicalMixing  (100% in ALL seasons)")
print("Secondary bottleneck: GlueAndDrying   (96-99%)")
print("Tertiary bottleneck:  CNC             (94-96%)")

## 5. Visualisation — Utilization Heatmap

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

stations = df_util['Station']
fall_u   = df_util['Fall (%)']
spring_u = df_util['Spring (%)']
winter_u = df_util['Winter (%)']

x = np.arange(len(stations))
width = 0.25

def util_color(u):
    if u >= 99: return '#e74c3c'
    elif u >= 90: return '#f39c12'
    elif u >= 80: return '#f1c40f'
    else: return '#2ecc71'

# Grouped bar chart
b1 = ax1.bar(x - width, fall_u,   width, label='Fall',   color=[util_color(u) for u in fall_u],   alpha=0.85, edgecolor='white')
b2 = ax1.bar(x,         spring_u, width, label='Spring', color=[util_color(u) for u in spring_u], alpha=0.65, edgecolor='white')
b3 = ax1.bar(x + width, winter_u, width, label='Winter', color=[util_color(u) for u in winter_u], alpha=0.50, edgecolor='white')

ax1.axhline(100, color='red', linewidth=1.5, linestyle='--', alpha=0.7, label='100% capacity')
ax1.set_xticks(x)
ax1.set_xticklabels([s.replace('And', '\n& ') for s in stations], fontsize=8, rotation=15, ha='right')
ax1.set_ylabel('Scheduled Utilization (%)')
ax1.set_title('Station Utilization by Season', fontsize=12, fontweight='bold')
ax1.legend()
ax1.set_ylim(0, 110)

red   = mpatches.Patch(color='#e74c3c', label='≥99% (Bottleneck)')
orange= mpatches.Patch(color='#f39c12', label='90-99% (High)')
yellow= mpatches.Patch(color='#f1c40f', label='80-90% (Moderate)')
green = mpatches.Patch(color='#2ecc71', label='<80% (Normal)')
ax1.legend(handles=[red, orange, yellow, green], fontsize=8, loc='lower right')

# Throughput gap chart
seasons = ['Fall', 'Spring', 'Winter']
demands = [200, 300, 40]
throughputs = [20, 20, 20]
gaps = [d - t for d, t in zip(demands, throughputs)]

x2 = np.arange(len(seasons))
ax2.bar(x2, demands, label='Demand', color='#3498db', alpha=0.7, width=0.4)
ax2.bar(x2, throughputs, label='Actual throughput', color='#e74c3c', alpha=0.9, width=0.4)
for i, (d, t) in enumerate(zip(demands, throughputs)):
    ax2.annotate(f'{t/d*100:.0f}%\nmet', xy=(i, t), xytext=(i, t + 8),
                 ha='center', fontsize=9, color='darkred', fontweight='bold')
ax2.set_xticks(x2)
ax2.set_xticklabels(seasons)
ax2.set_ylabel('Tubes per Day')
ax2.set_title('Throughput vs Demand by Season', fontsize=12, fontweight='bold')
ax2.legend()

plt.suptitle('Current State — Confetti Production Simulation Results', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('current_state_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. WIP and Time in System

In [ ]:
# Time in system and WIP from Simio ModelEntity population statistics
wip_data = pd.DataFrame({
    'Tube Type': ['Small', 'Large', 'XL'],
    'Avg. Time in System (hrs)': [320, 380, 350],
    'Avg. Time in System (days)': [round(t/8, 1) for t in [320, 380, 350]],
    'Max. Time in System (hrs)': [1245, 1265, 1255],
    'Avg. WIP (units)': [366, 203, 268]
})

print("WIP and Flow Time Statistics (Current State — Fall Scenario):")
print(wip_data.to_string(index=False))
print(f"\nTotal average WIP in system: {wip_data['Avg. WIP (units)'].sum()} tubes")
print(f"Max time in system: ~{1265/24:.0f} days — some tubes remain in system for almost the FULL 60-day run")
print(f"\nA system producing only 20 tubes/day while 837 tubes sit in WIP is a")
print(f"classic symptom of severe downstream bottlenecking and upstream blocking.")

## 7. Current vs Future State Comparison

In [ ]:
# Improvement interventions modeled in the future state
improvements = [
    ('Add server at ChemicalMixing',      'Primary bottleneck at 100% utilization — doubles capacity'),
    ('Add server at GlueAndDrying',       'Secondary bottleneck at 96-99% — prevents new bottleneck forming'),
    ('Extend shift: 480 → 540 min/day',   'Provides 12.5% more processing time across all stations'),
    ('WIP cap (CONWIP/Kanban logic)',      'Prevents upstream blocking — reduces avg time in system'),
]

print("Future State Improvements Modeled:")
print("-" * 70)
for i, (action, rationale) in enumerate(improvements, 1):
    print(f"{i}. {action}")
    print(f"   Rationale: {rationale}")
    print()

# Scenario comparison summary
comparison = pd.DataFrame({
    'Metric': [
        'Throughput — Fall (tubes/day)',
        'Throughput — Spring (tubes/day)',
        'Throughput — Winter (tubes/day)',
        'ChemicalMixing utilization (%)',
        'GlueAndDrying utilization (%)',
        'Avg. time in system (hrs)',
        'Avg. WIP (total units)',
        'Demand met — Fall (%)',
    ],
    'Current State': [20, 20, 20, '100 (bottleneck)', '99.1', '300–400', '~837', '10%'],
    'Future State':  [65, 70, 38, '~75', '~82', '~120–160', '~280', '32%'],
    'Improvement': ['+225%', '+250%', '+90%', 'Relieved', 'Relieved', '~60% reduction', '~67% reduction', '+22 pp']
})

print("\nCurrent vs Future State Summary:")
print(comparison.to_string(index=False))

## 8. Key Findings & Methodology Notes

### Bottleneck Analysis
The primary bottleneck is **ChemicalMixing** (100% utilization in every season), not LabelingAndPackaging as initially hypothesized. Secondary bottlenecks are **GlueAndDrying** (96–99%) and **CNC** (94–96%).

### Root Cause of Throughput Failure
The system produces only **~20 tubes/day** regardless of whether demand is 40 (Winter) or 300 (Spring). This throughput invariance is the definitive signal of a binding capacity constraint — the line is limited by its slowest stage, not by incoming orders.

### Material Availability
No material shortages occurred over the 60-day horizon. Raw material supply is **not** the constraint — the system's physical processing capacity is the binding limit.

### Statistical Design
- **15 replications** per scenario for statistical reliability at 95% confidence
- **Triangular distributions** for service times (estimated from GEMBA walk videos)
- **Blocking detection** via custom state variables (`BlockedCount`, `BlockedWaitTime`)
- **Table-driven architecture** (ProcessSequenceTable) for maintainability

### Model Files
| File | Description |
|------|-------------|
| `Current_State_Final_Model.spfx` | Baseline model — current production system |
| `Future_State_Final_Model.spfx` | Improved model — added capacity + shift extension |
| `Simio_Final_Model.spfx` | Combined/final submission model |

> Open `.spfx` files in [Simio](https://www.simio.com/) (free academic version available). Full written analysis is in `Simulation_Report.pdf`.